# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the Mass Spectrometry EV dataset using `mlcroissant`.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running in a new environment)
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print summary
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print("\nKeywords:", dataset.metadata.keywords)
print("\nCitation (citeAs):", getattr(dataset.metadata, 'citeAs', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs/structure.

In [ ]:
# List available record sets with their @id, name, and associated fields
record_sets = dataset.metadata.recordSet
if not isinstance(record_sets, list):
    record_sets = [record_sets] if record_sets else []

print('Discovered record sets:')
recordset_info = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None) or None
    name = getattr(rs, 'name', None)
    fields = getattr(rs, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    field_ids = [(getattr(f, '@id', None) or getattr(f, 'id', None)) for f in fields]
    print(f"- RecordSet @id: {rs_id} | name: {name} | #fields: {len(fields)}")
    recordset_info.append({'id': rs_id, 'name': name, 'field_ids': field_ids})

if len(recordset_info) == 0:
    print("No explicit record sets were found in the metadata. Attempting to auto-detect via dataset.records()...")
    # Try to enumerate all record sets
    all_ids = dataset.list_record_sets()
    print('Detected possible record set ids:', all_ids)

## 3. Data Extraction
Extract records from the main record set into a DataFrame and inspect available columns.

**Note:** All references to dataset entities, fields, and columns use their `@id`.

In [ ]:
# Use dataset.list_record_sets() to enumerate RecordSets
record_set_ids = dataset.list_record_sets()
print(f"Record sets detected via mlcroissant: {record_set_ids}")

dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records for RecordSet {rsid}")
    else:
        print(f"No data found for RecordSet {rsid}")

if len(dataframes) > 0:
    # Pick the main record set for downstream analysis: heuristically pick the one with most records
    main_record_set_id = max(dataframes.keys(), key=lambda k: len(dataframes[k]))
    main_df = dataframes[main_record_set_id]
    print(f"Main RecordSet selected: {main_record_set_id}")
else:
    print('No DataFrames extracted; check dataset schema and files.')

### Inspect Columns of the Main RecordSet
Column/field names are discovered from the dataframe loaded from the primary record set.

In [ ]:
if 'main_df' in locals():
    print('Columns available in main record set:')
    print(main_df.columns.tolist())
    display(main_df.head())
else:
    print('No main DataFrame available.')

## 4. Exploratory Data Analysis (EDA)
Apply common data preprocessing and filtering steps. This section shows filtering by a numeric field, normalization, and grouping.

Some likely numeric fields in proteomics/EV datasets include `MW`, `Coverage Percent`, `Peptide Counts`, or normalized abundance values. Use available column `@id`s from the loaded DataFrame.

In [ ]:
# --- Example EDA ---
if 'main_df' in locals():
    # Attempt to select a numeric field for demonstration: prefer 'MW', 'normalized_abundance', etc.
    numeric_fields = [c for c in main_df.columns if main_df[c].dtype.kind in 'ifu']
    print('Numeric candidate fields:', numeric_fields)

    # Fallback: try to find fields with known names
    preferred_numeric_fields = ['MW', 'molecular_weight', 'Coverage Percent', 'coverage', 'normalized_abundance', 'peptide_count', 'Peptide Counts']
    chosen_field = None
    for pf in preferred_numeric_fields:
        for c in main_df.columns:
            if pf.lower().replace(' ', '') in c.lower().replace(' ', ''):
                chosen_field = c
                break
        if chosen_field:
            break
    if not chosen_field and numeric_fields:
        chosen_field = numeric_fields[0]
    if chosen_field:
        print(f"Chosen numeric field for analysis: {chosen_field}")
        # Try normalization and filtering
        try:
            main_df[chosen_field] = pd.to_numeric(main_df[chosen_field], errors='coerce')
            thr = main_df[chosen_field].quantile(0.75)  # 75th percentile as threshold
            filtered_df = main_df[main_df[chosen_field] > thr].copy()
            print(f"Filtered {len(filtered_df)} records with {chosen_field} > {thr:.2f}")
            norm_col = f"{chosen_field}_normalized"
            filtered_df[norm_col] = (filtered_df[chosen_field] - filtered_df[chosen_field].mean()) / filtered_df[chosen_field].std()
            display(filtered_df[[chosen_field, norm_col]].head())

            # Grouping (if possible)
            group_field = None
            for gf in ['category', 'sample', 'Sample', 'group', 'Group']:
                for c in main_df.columns:
                    if gf.lower() in c.lower():
                        group_field = c
                        break
                if group_field:
                    break
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[chosen_field].mean()
                print(f"Grouped mean of {chosen_field} by {group_field}:")
                display(grouped_df.head())
            else:
                print('No suitable group field was found for grouping.')
        except Exception as e:
            print(f"Numeric analysis failed: {e}")
    else:
        print("No numeric fields were found for analysis.")
else:
    print('Main DataFrame unavailable for analysis.')

## 5. Visualization
Visualize distributions of important numeric fields or show relationships among several fields.

We'll show the distribution of the chosen numeric field and any group-wise means (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if 'main_df' in locals() and chosen_field:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[chosen_field].dropna(), kde=True)
    plt.title(f'Distribution of {chosen_field}')
    plt.xlabel(chosen_field)
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=main_df[group_field], y=main_df[chosen_field])
        plt.title(f'{chosen_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No suitable numeric field for visualization.')

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load a Croissant-annotated dataset using `mlcroissant`;
- Auto-discover record sets and explore the dataset structure using `@id` references;
- Extract tables into Pandas and examine relevant fields;
- Apply simple filters and normalizations to key protein attributes;
- Visualize distributions to understand field characteristics.

For advanced analysis, reference MLCommons Croissant documentation and extend this notebook to aggregate or join across related RecordSets using their `@id` links.